# Demo 01 : Graphs and state machines

Run the cells in order and watch. There is nothing to write.

## a · directed-graphs

A game build at Comet Arcade has three steps. Each step becomes a node, and each edge says which step starts when the
last one finishes. The nodes are plain functions for now, so the graph runs without a model. They write fixed text
onto the project board.

In [1]:
from typing import TypedDict
from langgraph.graph import END, START, StateGraph

class Board(TypedDict):
    brief: str
    rules: str
    code: str
    bugs: list[str]

def designer(board: Board) -> dict:
    return {"rules": "eat stars, dodge comets"}

def coder(board: Board) -> dict:
    return {"code": "game.html"}

def tester(board: Board) -> dict:
    return {"bugs": []}

builder = StateGraph(Board)
builder.add_node("designer", designer)
builder.add_node("coder", coder)
builder.add_node("tester", tester)
builder.add_edge(START, "designer")
builder.add_edge("designer", "coder")
builder.add_edge("coder", "tester")
builder.add_edge("tester", END)
graph = builder.compile()
print(graph.invoke({"brief": "snake in space", "rules": "", "code": "", "bugs": []}))
print(graph.get_graph().draw_mermaid())

{'brief': 'snake in space', 'rules': 'eat stars, dodge comets', 'code': 'game.html', 'bugs': []}
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	designer(designer)
	coder(coder)
	tester(tester)
	__end__([<p>__end__</p>]):::last
	__start__ --> designer;
	coder --> tester;
	designer --> coder;
	tester --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## b · fix-loop

We add the fix loop. After the tests a router sends the build to `ship`, back to the Coder, or out of the graph at the
cap of 3 attempts. The input key `passes_on` stands in for a real Coder : it says on which attempt the tests pass.
The cap is a key on the board, so the loop ends even when no attempt ever passes.

In [2]:
from typing import Literal

class Build(TypedDict):
    passes_on: int
    status: str
    attempts: int
    history: list[str]

def coder(board: Build) -> dict:
    attempts = board["attempts"] + 1
    status = f"build {attempts}"
    return {"status": status, "attempts": attempts, "history": [*board["history"], status]}

def tester(board: Build) -> dict:
    status = "tests pass" if board["attempts"] >= board["passes_on"] else "tests fail"
    return {"status": status, "history": [*board["history"], status]}

def route(board: Build) -> Literal["ship", "coder", "__end__"]:
    if board["status"] == "tests pass":
        return "ship"
    if board["attempts"] >= 3:
        return END
    return "coder"

def ship(board: Build) -> dict:
    return {"status": "shipped", "history": [*board["history"], "shipped"]}

loop = StateGraph(Build)
loop.add_node("coder", coder)
loop.add_node("tester", tester)
loop.add_node("ship", ship)
loop.add_edge(START, "coder")
loop.add_edge("coder", "tester")
loop.add_conditional_edges("tester", route)
loop.add_edge("ship", END)
fix_loop = loop.compile()

for passes_on in (2, 99):
    result = fix_loop.invoke({"passes_on": passes_on, "status": "brief", "attempts": 0, "history": []})
    print("history:", result["history"])
    print("status:", result["status"], "attempts:", result["attempts"])

history: ['build 1', 'tests fail', 'build 2', 'tests pass', 'shipped']
status: shipped attempts: 2
history: ['build 1', 'tests fail', 'build 2', 'tests fail', 'build 3', 'tests fail']
status: tests fail attempts: 3


## Things to try

- Add an `artist` node between the Designer and the Coder
- Lower the cap to 2 and run the build that never passes